In [1]:
import fiftyone as fo
import fiftyone.brain as fob

SOURCE_DATASETS =[
    "cip",
    "cubic",
    "rod",
    "spiky",
    "core_shell"
]
DEST_DATASET = "merged_dataset"
compute_embeddings = True

In [2]:

if fo.dataset_exists(DEST_DATASET):
    fo.delete_dataset(DEST_DATASET)
    dest = fo.Dataset(DEST_DATASET)
else:
    dest = fo.Dataset(DEST_DATASET)

seen = set() #to track seen filepaths
total_added = 0 #to track total samples added from source dataset into the merged dataset


You are running the oldest supported major version of MongoDB. Please refer to https://deprecation.voxel51.com for deprecation notices. You can suppress this exception by setting your `database_validation` config parameter to `False`. See https://docs.voxel51.com/user_guide/config.html#configuring-a-mongodb-connection for more information


In [3]:

for name in SOURCE_DATASETS:
    if not fo.dataset_exists(name):
        print(f"Source dataset '{name}' does not exist. Skipping merge.")
        continue

    src_dataset = fo.load_dataset(name)
    new_samples = []
    for sample in src_dataset:
        if sample.filepath in seen:
            continue
        new_samples.append(sample.copy()) #copy sample to avoid modifying original
        seen.add(sample.filepath)
    if new_samples:
        dest.add_samples(new_samples)
        total_added += len(new_samples)
        print(f"Added {len(new_samples)} samples from '{name}' to '{DEST_DATASET}'.") #log progress


 100% |█████████████████| 600/600 [139.8ms elapsed, 0s remaining, 4.3K samples/s]    
Added 600 samples from 'cip' to 'merged_dataset'.
 100% |███████████████████| 52/52 [13.9ms elapsed, 0s remaining, 3.7K samples/s]      
Added 52 samples from 'cubic' to 'merged_dataset'.
 100% |███████████████████| 52/52 [14.5ms elapsed, 0s remaining, 3.6K samples/s]      
Added 52 samples from 'rod' to 'merged_dataset'.
 100% |███████████████████| 28/28 [10.1ms elapsed, 0s remaining, 2.8K samples/s]     
Added 28 samples from 'spiky' to 'merged_dataset'.
 100% |███████████████████| 20/20 [8.4ms elapsed, 0s remaining, 2.4K samples/s]       
Added 20 samples from 'core_shell' to 'merged_dataset'.


In [7]:
#USING CLIP VIT BASE32 TORCH MODEL
dest.persistent = True
print(f"Merging complete. Total samples in '{DEST_DATASET}': {dest.count()}, added {total_added} new samples.")
if compute_embeddings:
    fob.compute_visualization(
        dest,
        #embeddings="resnet50-emb",
        brain_key="gt_viz",
        model="clip-vit-base32-torch",
        method="umap",
        create_index=True,  # Must be false for 3d
        force_recompute=False,
        num_dims=2,
        seed=42,
        min_dist=0.2
    )

Merging complete. Total samples in 'merged_dataset': 48, added 48 new samples.
Computing embeddings...
 100% |███████████████████| 48/48 [18.4s elapsed, 0s remaining, 2.8 samples/s]      
Generating visualization...
UMAP(min_dist=0.2, n_jobs=1, random_state=42, verbose=True)
Tue Sep 23 12:41:55 2025 Construct fuzzy simplicial set
Tue Sep 23 12:41:55 2025 Finding Nearest Neighbors
Tue Sep 23 12:41:55 2025 Finished Nearest Neighbor Search
Tue Sep 23 12:41:55 2025 Construct embedding


C:\Users\ERFIGO\miniconda3\envs\fiftyone\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Epochs completed:   0%|            0/500 [00:00]

	completed  0  /  500 epochs
	completed  50  /  500 epochs
	completed  100  /  500 epochs
	completed  150  /  500 epochs
	completed  200  /  500 epochs
	completed  250  /  500 epochs
	completed  300  /  500 epochs
	completed  350  /  500 epochs
	completed  400  /  500 epochs
	completed  450  /  500 epochs
Tue Sep 23 12:41:55 2025 Finished embedding
Generating spatial index in field 'gt_viz'...
 100% |███████████████████| 48/48 [13.0ms elapsed, 0s remaining, 3.7K samples/s]      


In [4]:
#USING RESNET MODELS
dest.persistent = True
print(f"Merging complete. Total samples in '{DEST_DATASET}': {dest.count()}, added {total_added} new samples.")
if compute_embeddings:
    fob.compute_visualization(
        dest,
        #embeddings="resnet50-emb",
        brain_key="rn50_umap",
        model="resnet50-imagenet-torch",
        method="umap",
        create_index=True,  # Must be false for 3d
        force_recompute=False,
        num_dims=2,
        min_dist=0.2
    )

Merging complete. Total samples in 'merged_dataset': 752, added 752 new samples.
Computing embeddings...
 100% |█████████████████| 752/752 [24.4m elapsed, 0s remaining, 0.2 samples/s]    
Generating visualization...
UMAP(min_dist=0.2, verbose=True)
Tue Sep 23 13:49:48 2025 Construct fuzzy simplicial set
Tue Sep 23 13:49:49 2025 Finding Nearest Neighbors
Tue Sep 23 13:49:54 2025 Finished Nearest Neighbor Search
Tue Sep 23 13:49:57 2025 Construct embedding


Epochs completed:   0%|            0/500 [00:00]

	completed  0  /  500 epochs
	completed  50  /  500 epochs
	completed  100  /  500 epochs
	completed  150  /  500 epochs
	completed  200  /  500 epochs
	completed  250  /  500 epochs
	completed  300  /  500 epochs
	completed  350  /  500 epochs
	completed  400  /  500 epochs
	completed  450  /  500 epochs
Tue Sep 23 13:49:59 2025 Finished embedding
Generating spatial index in field 'rn50_umap'...
 100% |█████████████████| 752/752 [72.8ms elapsed, 0s remaining, 10.3K samples/s]  


In [5]:
info = fo.load_dataset(DEST_DATASET).get_brain_info("rn50_umap")
print(info)

{
    "key": "rn50_umap",
    "version": "1.8.0",
    "timestamp": "2025-09-23T11:25:13.345000",
    "config": {
        "cls": "fiftyone.brain.visualization.UMAPVisualizationConfig",
        "type": "visualization",
        "method": "umap",
        "embeddings_field": null,
        "points_field": "rn50_umap",
        "similarity_index": null,
        "model": "resnet50-imagenet-torch",
        "model_kwargs": null,
        "patches_field": null,
        "num_dims": 2,
        "force_recompute": false,
        "num_neighbors": 15,
        "metric": "euclidean",
        "min_dist": 0.2,
        "seed": null,
        "verbose": true
    }
}


In [6]:
session = fo.launch_app(dest)
session.wait()
session.url

Connected to FiftyOne on port 5151 at localhost.
If you are not connecting to a remote session, you may need to start a new session and specify a port


Notebook sessions cannot wait


'http://localhost:5151/'

In [ ]:
"""git initalization"""

In [1]:
!git init

Reinitialized existing Git repository in C:/Users/ERFIGO/Documents/fiftyoneproject-master/.git/


In [5]:
!git status

On branch branch1-embedding
Your branch is up to date with 'fiftyone/branch1-embedding'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   embeddings_updated.ipynb
	modified:   fiftyone_additional_features.ipynb
	modified:   fiftyone_embedding.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.ipynb_checkpoints/

no changes added to commit (use "git add" and/or "git commit -a")


In [3]:
!git add embeddings_updated.ipynb .gitignore

In [4]:
!git commit -m "feat: adding clip vit base32 torch as comparison model"

[branch1-embedding e28da6b] feat: adding embeddings updated for more focused embeddings research
 1 file changed, 156 insertions(+)
 create mode 100644 embeddings_updated.ipynb


In [5]:
!git push

To https://github.com/erfigo/fiftyoneproject.git
   d5e3a74..e28da6b  branch1-embedding -> branch1-embedding
